# 01 · Prepare Docker historical tickets (Tier 2)

Pulls two Hugging Face datasets, keeps only **Docker-related** rows, and writes one consolidated,
de-duplicated ticket file.

| Source | Rows | Columns | Notes |
|---|---|---|---|
| `pavanmantha/devops-v1` | 312 | `question`, `solution` | Short Q&A with real error strings; card gives no origin |
| `sequelbox/Titanium` | 26,577 | `prompt`, `response` | Synthetic (Llama-3.1-405B), not manually reviewed; mostly "how do I / what is" prompts with long essay answers |

Outputs (in `OUT_ROOT`): `raw/*.jsonl` (untouched copies), `processed/docker_tickets.jsonl`,
`processed/docker_tickets.csv`, `processed/docker_tickets_report.json`.

Run top to bottom. Nothing here needs an API key.

In [ ]:
# ---- Config -------------------------------------------------------------------------
# Priority order matters: if the same problem appears in both sources, the earlier one wins.
SOURCES = [
    {"name": "pavanmantha/devops-v1", "problem_col": "question", "resolution_col": "solution",
     "provenance": "unspecified"},
    {"name": "sequelbox/Titanium", "problem_col": "prompt", "resolution_col": "response",
     "provenance": "llm_synthetic"},
]

INCLUDE_SECONDARY = False   # True also keeps rows where "docker" appears only in the answer
MIN_PROBLEM_CHARS = 15      # drop fragments too short to be a real problem statement
USE_DRIVE = True            # save under Google Drive so outputs survive the Colab session

!pip install -q datasets

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = Path("/content/drive/MyDrive/customer-support-triage/tickets")
    except ImportError:
        OUT_ROOT = Path("tickets")
else:
    OUT_ROOT = Path("tickets")

RAW_DIR = OUT_ROOT / "raw"
PROCESSED_DIR = OUT_ROOT / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Writing to:", OUT_ROOT.resolve())

## 1 · Load both datasets

The raw rows are saved untouched, and each dataset's commit SHA is recorded so the run is reproducible.

In [ ]:
from datasets import load_dataset


def dataset_revision(name):
    try:
        from huggingface_hub import HfApi
        return HfApi().dataset_info(name).sha
    except Exception:
        return "unknown"


def load_source(src):
    df = load_dataset(src["name"], split="train").to_pandas()
    df.to_json(RAW_DIR / (src["name"].replace("/", "__") + ".jsonl"),
               orient="records", lines=True, force_ascii=False)

    df = df.rename(columns={src["problem_col"]: "problem", src["resolution_col"]: "resolution"})
    df["source"] = src["name"]
    df["source_row"] = range(len(df))
    df["provenance"] = src["provenance"]
    return df[["source", "source_row", "provenance", "problem", "resolution"]]


revisions = {src["name"]: dataset_revision(src["name"]) for src in SOURCES}
frames = [load_source(src) for src in SOURCES]
tickets = pd.concat(frames, ignore_index=True)

report = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "revisions": revisions,
          "loaded": tickets["source"].value_counts().to_dict()}
print(report["loaded"])

## 2 · Clean text

Whitespace is normalised (newlines kept so code blocks survive). Emails and `password=...`-style secrets
are masked; IP addresses are deliberately left alone because Docker answers are full of `0.0.0.0` and
`127.0.0.1` examples.

In [ ]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
SECRET_RE = re.compile(r"(?i)\b(password|passwd|token|secret|api[_-]?key)\b(\s*[:=]\s*)\S+")


def clean_text(value):
    text = "" if value is None else str(value)
    text = text.replace("\r\n", "\n").strip()
    text = re.sub(r"[ \t]+", " ", text)
    text = EMAIL_RE.sub("<email>", text)
    return SECRET_RE.sub(r"\1\2<redacted>", text)


tickets["problem"] = tickets["problem"].map(clean_text)
tickets["resolution"] = tickets["resolution"].map(clean_text)

before = len(tickets)
tickets = tickets[(tickets["problem"].str.len() >= MIN_PROBLEM_CHARS) & (tickets["resolution"] != "")]
report["dropped_empty_or_short"] = before - len(tickets)
print("dropped empty/short:", report["dropped_empty_or_short"])

## 3 · Keep only Docker-related rows

* **primary** – "docker" appears in the problem itself (kept)
* **secondary** – "docker" appears only in the answer, e.g. a Kubernetes or Jenkins question that mentions it in passing (dropped unless `INCLUDE_SECONDARY`)

In [ ]:
DOCKER_RE = re.compile(r"docker", re.I)


def docker_relevance(row):
    if DOCKER_RE.search(row["problem"]):
        return "primary"
    if DOCKER_RE.search(row["resolution"]):
        return "secondary"
    return "none"


tickets["relevance"] = tickets.apply(docker_relevance, axis=1)
report["relevance_by_source"] = (
    tickets.groupby(["source", "relevance"]).size().unstack(fill_value=0).to_dict("index"))
print(pd.DataFrame(report["relevance_by_source"]).T)

keep = {"primary", "secondary"} if INCLUDE_SECONDARY else {"primary"}
tickets = tickets[tickets["relevance"].isin(keep)].copy()
print("kept after Docker filter:", len(tickets))

## 4 · Enrich

Cheap, rule-based fields that later stages can use:

* `kind_guess` – `troubleshooting` / `how_to` / `concept`, from the wording of the problem (statements and error text count as troubleshooting; "what is / how do I" questions do not). Titanium is mostly the latter, so this lets you subset. It is a heuristic, not a label.
* `topics` – named after the Docker docs areas (`build`, `compose`, `engine`, `networking`, `storage`, `registry`, `desktop`, `security`, `runtime`, `swarm`) so they line up with the KB's `product_area` later.
* `error_strings`, `exit_codes` – quoted error text and exit codes found in the problem. Good BM25 fodder.

In [ ]:
TROUBLE_RE = re.compile(
    r"\b(error|fail(s|ed|ing|ure)?|not working|cannot|can't|unable|issue|problem|troubleshoot|debug|"
    r"crash(es|ed)?|denied|time(d)? ?out|refused|stuck|exits?|exited|hangs?|not found|no space)\b", re.I)
HOWTO_RE = re.compile(r"^\s*(how (do|can|to|should)|what (is|are) the (command|steps?|way)|steps? to)", re.I)
QUESTION_RE = re.compile(
    r"^\s*(how|what|why|which|when|where|who|can|could|does|do|is|are|should|would|explain|describe|"
    r"compare|list|write|provide|give|show|create|design|walk)\b", re.I)
QUOTED_RE = re.compile(r"['\"]([^'\"\n]{12,200})['\"]")
EXIT_RE = re.compile(r"\bexit(?:ed)?(?: with)?(?: code| status)?\s*(\d{1,3})\b", re.I)

TOPIC_PATTERNS = {
    "build": r"dockerfile|docker build|buildx|buildkit|multi-stage|build cache|\blayers?\b",
    "compose": r"compose",
    "engine": r"daemon|dockerd|systemctl|docker\.sock|docker\.pid|/etc/docker|cgroup",
    "networking": r"network|\bports?\b|\bdns\b|bridge|expose|connectivity",
    "storage": r"volume|bind mount|disk|no space|prune|overlay",
    "registry": r"docker hub|dockerhub|registry|\bpull\b|\bpush\b|docker login|rate limit|manifest",
    "desktop": r"docker desktop|\bwsl\b|hyper-v|macos|apple silicon",
    "security": r"permission denied|rootless|privileged|seccomp|apparmor|vulnerab|user namespace",
    "runtime": r"exits?|stops?|crash|\boom\b|memory|restart|healthcheck|exit code",
    "swarm": r"swarm|stack deploy",
}
TOPIC_RES = {topic: re.compile(pattern, re.I) for topic, pattern in TOPIC_PATTERNS.items()}


def find_error_strings(problem):
    return [m.group(1).strip() for m in QUOTED_RE.finditer(problem)]


def guess_kind(problem, error_strings):
    # Real problem reports tend to be statements ("Docker volume not persisting data."),
    # whereas instruct-style prompts are questions or commands.
    if error_strings or not QUESTION_RE.match(problem) or TROUBLE_RE.search(problem):
        return "troubleshooting"
    if HOWTO_RE.match(problem):
        return "how_to"
    return "concept"


tickets["error_strings"] = tickets["problem"].map(find_error_strings)
tickets["exit_codes"] = tickets["problem"].map(lambda p: sorted(set(EXIT_RE.findall(p))))
tickets["kind_guess"] = [guess_kind(p, e) for p, e in zip(tickets["problem"], tickets["error_strings"])]
tickets["topics"] = tickets["problem"].map(
    lambda p: [topic for topic, rx in TOPIC_RES.items() if rx.search(p)])
tickets["problem_words"] = tickets["problem"].str.split().str.len()
tickets["resolution_words"] = tickets["resolution"].str.split().str.len()

print(tickets.groupby(["source", "kind_guess"]).size().unstack(fill_value=0))

## 5 · De-duplicate

Exact match on the normalised problem text, within and across sources (earlier source in `SOURCES` wins).

In [ ]:
def problem_key(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


priority = {src["name"]: i for i, src in enumerate(SOURCES)}
tickets["problem_key"] = tickets["problem"].map(problem_key)
tickets["_priority"] = tickets["source"].map(priority)

before = len(tickets)
tickets = (tickets.sort_values(["_priority", "source_row"])
                  .drop_duplicates("problem_key", keep="first")
                  .drop(columns=["_priority", "problem_key"]))
report["dropped_duplicates"] = before - len(tickets)
print("dropped duplicates:", report["dropped_duplicates"], "| remaining:", len(tickets))

## 6 · Save

`ticket_id` is a hash of source + original row number, so it stays stable across re-runs.

In [ ]:
tickets["ticket_id"] = [
    "T-" + hashlib.sha1(f"{s}:{r}".encode()).hexdigest()[:12]
    for s, r in zip(tickets["source"], tickets["source_row"])
]

columns = ["ticket_id", "source", "source_row", "provenance", "problem", "resolution", "relevance",
           "kind_guess", "topics", "error_strings", "exit_codes", "problem_words", "resolution_words"]
tickets = tickets[columns].reset_index(drop=True)

jsonl_path = PROCESSED_DIR / "docker_tickets.jsonl"
tickets.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

csv_frame = tickets.copy()
for col in ["topics", "error_strings", "exit_codes"]:
    csv_frame[col] = csv_frame[col].map(json.dumps)
csv_frame.to_csv(PROCESSED_DIR / "docker_tickets.csv", index=False)

report.update({
    "config": {"include_secondary": INCLUDE_SECONDARY, "min_problem_chars": MIN_PROBLEM_CHARS},
    "final_count": len(tickets),
    "final_by_source": tickets["source"].value_counts().to_dict(),
    "final_by_kind": tickets["kind_guess"].value_counts().to_dict(),
    "final_by_topic": pd.Series([t for row in tickets["topics"] for t in row]).value_counts().to_dict(),
})
(PROCESSED_DIR / "docker_tickets_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

print(json.dumps({k: report[k] for k in ["final_count", "final_by_source", "final_by_kind"]}, indent=2))

## 7 · Spot-check

In [ ]:
pd.set_option("display.max_colwidth", 110)
for source, group in tickets.groupby("source"):
    print(f"\n=== {source} ({len(group)} rows)")
    for _, row in group.sample(min(3, len(group)), random_state=1).iterrows():
        print(f"[{row.kind_guess}] {row.problem[:160]}")
        print(f"    -> {row.resolution[:160].replace(chr(10), ' ')}")

In [ ]:
# Optional: download the outputs to your computer.
try:
    from google.colab import files
    for name in ["docker_tickets.jsonl", "docker_tickets.csv", "docker_tickets_report.json"]:
        files.download(str(PROCESSED_DIR / name))
except ImportError:
    pass